In [ ]:
import fastf1

session = fastf1.get_session(2021, 7, "Q")
print(session.name)
print(session.date)

In [ ]:
session.event

In [ ]:
schedule = fastf1.get_event_schedule(2021)
schedule

In [ ]:
session.load()
session.results

In [ ]:
session.load()
laps = session.laps

first_lap = laps.iloc[0]


In [ ]:
lap = laps.pick_fastest()
print(type(lap))


In [ ]:
fastest_lap = session.laps.pick_fastest()
print(fastest_lap["LapTime"])
print(fastest_lap["Driver"])


In [ ]:
weather_data = session.laps.get_weather_data()
weather_data

In [ ]:
fastest_lap = session.laps.pick_fastest()
print(fastest_lap["LapNumber"])
print(fastest_lap["LapTime"])

weather_for_fastest = fastest_lap.get_weather_data()
weather_for_fastest


In [ ]:
# Analise de completude por temporada para definir o escopo do MVP.
# Modo de triagem: uma corrida por evento e um limite de eventos por temporada
# evitam exceder limites da API. A analise completa pode ser habilitada depois.
from pathlib import Path

import fastf1
import matplotlib.pyplot as plt
import pandas as pd

CACHE_DIR = Path("../cache/fastf1")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

SEASONS = list(range(2018, 2025))
MAX_EVENTS_PER_SEASON = 3
LOAD_KWARGS = {"weather": True, "messages": False}


def _has_rows(frame):
    return frame is not None and not frame.empty


def inspect_session(year, round_number):
    row = {
        "season": year,
        "round": round_number,
        "session_type": "R",
        "loaded": False,
        "has_results": False,
        "has_laps": False,
        "has_weather": False,
        "result_rows": 0,
        "lap_rows": 0,
        "error": None,
    }
    try:
        session = fastf1.get_session(year, round_number, "R")
        session.load(**LOAD_KWARGS)
        row.update(
            loaded=True,
            has_results=_has_rows(session.results),
            has_laps=_has_rows(session.laps),
            result_rows=0 if session.results is None else len(session.results),
            lap_rows=0 if session.laps is None else len(session.laps),
        )
        if _has_rows(session.laps):
            row["has_weather"] = _has_rows(session.laps.get_weather_data())
    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"
    return row


records = []
for season in SEASONS:
    try:
        schedule = fastf1.get_event_schedule(season, include_testing=False)
        race_events = schedule[schedule["RoundNumber"].notna()]
        if MAX_EVENTS_PER_SEASON is None or len(race_events) <= MAX_EVENTS_PER_SEASON:
            selected = race_events
        else:
            positions = [
                round(index * (len(race_events) - 1) / (MAX_EVENTS_PER_SEASON - 1)) for index in range(MAX_EVENTS_PER_SEASON)
            ]
            selected = race_events.iloc[positions]
        for _, event in selected.iterrows():
            records.append(inspect_session(season, int(event["RoundNumber"])))
    except Exception as exc:
        records.append(
            {
                "season": season,
                "round": None,
                "session_type": "R",
                "loaded": False,
                "has_results": False,
                "has_laps": False,
                "has_weather": False,
                "result_rows": 0,
                "lap_rows": 0,
                "error": f"schedule: {type(exc).__name__}: {exc}",
            }
        )

session_audit = pd.DataFrame(records)
session_audit.head()

# Percentuais tornam temporadas com calendarios diferentes comparaveis.
season_summary = (
    session_audit.groupby("season")
    .agg(
        sampled_races=("round", "count"),
        loaded_sessions=("loaded", "sum"),
        sessions_with_results=("has_results", "sum"),
        sessions_with_laps=("has_laps", "sum"),
        sessions_with_weather=("has_weather", "sum"),
        total_result_rows=("result_rows", "sum"),
        total_lap_rows=("lap_rows", "sum"),
        failed_sessions=("error", lambda values: values.notna().sum()),
    )
    .reset_index()
)

for column in ("loaded_sessions", "sessions_with_results", "sessions_with_laps", "sessions_with_weather"):
    season_summary[f"{column}_pct"] = (100 * season_summary[column] / season_summary["sampled_races"].clip(lower=1)).round(1)

season_summary["completeness_score"] = (
    0.35 * season_summary["sessions_with_results_pct"]
    + 0.35 * season_summary["sessions_with_laps_pct"]
    + 0.15 * season_summary["sessions_with_weather_pct"]
    + 0.15 * season_summary["loaded_sessions_pct"]
).round(1)

season_summary = season_summary.sort_values(["completeness_score", "total_lap_rows"], ascending=False)
season_summary[
    [
        "season",
        "sampled_races",
        "loaded_sessions",
        "sessions_with_results",
        "sessions_with_laps",
        "sessions_with_weather",
        "total_lap_rows",
        "failed_sessions",
        "completeness_score",
    ]
]

plot_data = season_summary.sort_values("season")
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for column, label in (
    ("sessions_with_results_pct", "Resultados"),
    ("sessions_with_laps_pct", "Voltas"),
    ("sessions_with_weather_pct", "Clima"),
):
    axes[0].plot(plot_data["season"], plot_data[column], marker="o", label=label)
axes[0].set_title("Cobertura das corridas amostradas (%)")
axes[0].set_ylabel("Percentual das corridas")
axes[0].set_ylim(0, 105)
axes[0].legend()
axes[1].bar(plot_data["season"].astype(str), plot_data["total_lap_rows"], color="#d1495b")
axes[1].set_title("Volume de voltas na amostra")
axes[1].set_ylabel("Numero de registros de volta")
plt.tight_layout()
plt.show()

best_season = int(season_summary.iloc[0]["season"])
print(f"Temporada lider no screening: {best_season}")
print("Confirme o lider com MAX_EVENTS_PER_SEASON = None antes de congelar o MVP.")


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Driver 16 completed the race distance 00:00.050000 before the recorded end of the session.
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']


[seasons] rows=1
column dtype  nullable
season int64     False

[events] rows=22
           column          dtype  nullable
      RoundNumber          int64     False
          Country         object     False
         Location         object     False
OfficialEventName         object     False
        EventDate datetime64[ns]     False
        EventName         object     False
      EventFormat         object     False
         Session1         object     False
     Session1Date         object     False
  Session1DateUtc datetime64[ns]     False
         Session2         object     False
     Session2Date         object     False
  Session2DateUtc datetime64[ns]     False
         Session3         object     False
     Session3Date         object     False
  Session3DateUtc datetime64[ns]     False
         Session4         object     False
     Session4Date         object     False
  Session4DateUtc datetime64[ns]     False
         Session5         object     False
     Session5Dat